In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [8]:
wines = pd.read_csv('data/winemag-data_first150k.csv')
wines

,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,variety,winery
0,0,US,This tremendous 100% varietal wine hails from ...,Martha's Vineyard,96,235.0,California,Napa Valley,Napa,Cabernet Sauvignon,Heitz
1,1,Spain,"Ripe aromas of fig, blackberry and cassis are ...",Carodorum Selección Especial Reserva,96,110.0,Northern Spain,Toro,NaN,Tinta de Toro,Bodega Carmen Rodríguez
2,2,US,Mac Watson honors the memory of a wine once ma...,Special Selected Late Harvest,96,90.0,California,Knights Valley,Sonoma,Sauvignon Blanc,Macauley
3,3,US,"This spent 20 months in 30% new French oak, an...",Reserve,96,65.0,Oregon,Willamette Valley,Willamette Valley,Pinot Noir,Ponzi
4,4,France,"This is the top wine from La Bégude, named aft...",La Brûlade,95,66.0,Provence,Bandol,NaN,Provence red blend,Domaine de la Bégude
...,...,...,...,...,...,...,...,...,...,...,...
150925,150925,Italy,Many people feel Fiano represents southern Ita...,NaN,91,20.0,Southern Italy,Fiano di Avellino,NaN,White Blend,Feudi di San Gregorio
150926,150926,France,"Offers an intriguing nose with ginger, lime an...",Cuvée Prestige,91,27.0,Champagne,Champagne,NaN,Champagne Blend,H.Germain
150927,150927,Italy,This classic example comes from a cru vineyard...,Terre di Dora,91,20.0,Southern Italy,Fiano di Avellino,NaN,White Blend,Terredora
150928,150928,France,"A perfect salmon shade, with scents of peaches...",Grand Brut Rosé,90,52.0,Champagne,Champagne,NaN,Champagne Blend,Gosset


In [6]:
print(len(set(wines['title'])))

118840


In [5]:
import requests
import json

url = "https://www.vivino.com/api/explore/explore"
params = {
    "country_codes[]": "US",
    "min_rating": 3.5,
    "page": 1,
}
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json",
}

response = requests.get(url, params=params, headers=headers)
print(f"Status: {response.status_code}")

# Print the entire raw response so we can see exactly what Vivino is sending back
print("\nFull raw response:")
print(json.dumps(response.json(), indent=2))

Status: 404

Full raw response:


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [4]:
"""
test_search_enrichment.py
--------------------------
Tests the Brave Search enrichment pipeline end to end.
Run with:  python test_search_enrichment.py

Requires BRAVE_API_KEY to be set in your environment or .env file.
Each test uses 2 Brave API calls (1 image + 1 web search).
4 tests = 8 API calls total against your daily quota.
"""
import nest_asyncio
nest_asyncio.apply()
import asyncio
import os
from dotenv import load_dotenv

# Load .env so BRAVE_API_KEY is available
load_dotenv()

from backend.search_endpoint import enrich_wine, enrich_wines_batch, BRAVE_API_KEY


def check_key():
    if not BRAVE_API_KEY:
        print("ERROR: BRAVE_API_KEY is not set.")
        print("Add it to your .env file:  BRAVE_API_KEY=your_key_here")
        return False
    masked = BRAVE_API_KEY[:6] + "..." + BRAVE_API_KEY[-4:]
    print(f"BRAVE_API_KEY loaded: {masked}\n")
    return True


async def test_well_known_wine():
    """A famous wine — should return strong image and retail link results."""
    print("=" * 55)
    print("Test 1: Well-known wine (Caymus Cabernet Sauvignon)")
    print("=" * 55)
    result = await enrich_wine(
        wine_name="Cabernet Sauvignon",
        winery="Caymus Vineyards",
        region="Napa Valley",
        country="US",
    )
    print(f"  Wine:          {result.wine_name}")
    print(f"  Thumbnail URL: {result.thumbnail_url}")
    print(f"  Image source:  {result.image_source}")
    print(f"  Buy URL:       {result.buy_url}")
    print(f"  Buy source:    {result.buy_source}")

    # Buy URL is critical; thumbnail is enrichment so we warn not fail
    assert result.buy_url is not None, "Expected a buy URL"
    assert result.buy_url.startswith("https://"), "Buy URL should be HTTPS"
    if result.thumbnail_url:
        assert result.thumbnail_url.startswith("https://"), "Thumbnail should be HTTPS"
        print("  ✓ Passed (image + buy link)\n")
    else:
        print("  ⚠ Passed with warning: buy link found but no thumbnail\n")


async def test_old_world_wine():
    """A French wine — tests that non-US wines also return good results."""
    print("=" * 55)
    print("Test 2: Old World wine (Bordeaux)")
    print("=" * 55)
    result = await enrich_wine(
        wine_name="Margaux",
        winery="Château Margaux",
        region="Bordeaux",
        country="France",
    )
    print(f"  Wine:          {result.wine_name}")
    print(f"  Thumbnail URL: {result.thumbnail_url}")
    print(f"  Image source:  {result.image_source}")
    print(f"  Buy URL:       {result.buy_url}")
    print(f"  Buy source:    {result.buy_source}")

    assert result.thumbnail_url is not None, "Expected a thumbnail URL"
    assert result.buy_url is not None, "Expected a buy URL"
    print("  ✓ Passed\n")


async def test_obscure_wine():
    """
    A less-known wine — tests graceful degradation when results are sparse.
    We don't assert on thumbnail/buy_url here since they may genuinely
    not exist. We just confirm the function returns without crashing.
    """
    print("=" * 55)
    print("Test 3: Obscure wine (graceful degradation)")
    print("=" * 55)
    result = await enrich_wine(
        wine_name="Grüner Veltliner Smaragd",
        winery="Weingut Hirsch",
        region="Kamptal",
        country="Austria",
    )
    print(f"  Wine:          {result.wine_name}")
    print(f"  Thumbnail URL: {result.thumbnail_url or '(none returned)'}")
    print(f"  Buy URL:       {result.buy_url or '(none returned)'}")
    # Just confirm it didn't crash — no hard assertions on content
    assert isinstance(result.thumbnail_url, (str, type(None)))
    assert isinstance(result.buy_url, (str, type(None)))
    print("  ✓ Passed (no crash, fields may be None for obscure wines)\n")


async def test_batch_enrichment():
    """Tests the batch helper with 2 wines, confirming ordering is preserved."""
    print("=" * 55)
    print("Test 4: Batch enrichment (2 wines)")
    print("=" * 55)
    wines = [
        {"wine_name": "Pinot Noir", "winery": "Meiomi", "region": "California", "country": "US"},
        {"wine_name": "Malbec",     "winery": "Catena Zapata", "region": "Mendoza", "country": "Argentina"},
    ]
    results = await enrich_wines_batch(wines)

    assert len(results) == 2, f"Expected 2 results, got {len(results)}"
    assert results[0].wine_name == "Pinot Noir"
    assert results[1].wine_name == "Malbec"

    for r in results:
        print(f"  {r.wine_name}")
        print(f"    Thumbnail: {r.thumbnail_url}")
        print(f"    Buy link:  {r.buy_url} ({r.buy_source})")

    print("  ✓ Passed\n")


async def run_all():
    if not check_key():
        return

    print("Running Brave Search enrichment tests...\n")
    print(f"Total API calls this run: ~8 (2 per test × 4 tests)\n")

    await test_well_known_wine()
    await test_old_world_wine()
    await test_obscure_wine()
    await test_batch_enrichment()

    print("=" * 55)
    print("  All tests passed ✓")
    print("=" * 55)


if __name__ == "__main__":
    asyncio.run(run_all())

BRAVE_API_KEY loaded: BSAZig...pG_1

Running Brave Search enrichment tests...

Total API calls this run: ~8 (2 per test × 4 tests)

Test 1: Well-known wine (Caymus Cabernet Sauvignon)
[brave:image] HTTP 422 — query: 'Caymus Vineyards Cabernet Sauvignon Napa Valley wine bottle'
  Wine:          Cabernet Sauvignon
  Thumbnail URL: None
  Image source:  None
  Buy URL:       https://www.totalwine.com/wine/red-wine/cabernet-sauvignon/caymus-cabernet-sauvignon/p/223968750
  Buy source:    www.totalwine.com
  ⚠ Passed with warning: buy link found but no thumbnail

Test 2: Old World wine (Bordeaux)
[brave:image] HTTP 422 — query: 'Château Margaux Margaux Bordeaux wine bottle'
  Wine:          Margaux
  Thumbnail URL: None
  Image source:  None
  Buy URL:       https://www.totalwine.com/wine/brand/chateau-margaux
  Buy source:    www.totalwine.com


AssertionError: Expected a thumbnail URL